In [23]:
import requests
import json
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://www.nseindia.com/"
}

In [3]:
nif500 = r"NIFTY500SYM.csv"
df = pd.read_csv(nif500)
df.columns

Index(['Company Name', 'Industry', 'Symbol', 'Series', 'ISIN Code'], dtype='object')

In [5]:
df.Industry.unique()

array(['Financial Services', 'Diversified', 'Capital Goods',
       'Construction Materials', 'Power', 'Fast Moving Consumer Goods',
       'Chemicals', 'Healthcare', 'Metals & Mining', 'Services',
       'Oil Gas & Consumable Fuels', 'Consumer Services', 'Realty',
       'Construction', 'Information Technology',
       'Automobile and Auto Components', 'Consumer Durables',
       'Telecommunication', 'Textiles',
       'Media Entertainment & Publication'], dtype=object)

In [4]:
df.head(4)

,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025


In [9]:
symbols = df.Symbol.to_list()
len(symbols)

500

In [ ]:
base = {
 "index":"equities",
 "symbol":""   
}
    
session = requests.Session()
session.headers.update(headers)
session.get("https://www.nseindia.com", timeout=10, verify=False)
url = "https://www.nseindia.com/api/annual-reports"


report_list = []
for i,sym in enumerate(symbols):
    print(f"{i} : {sym}")
    base["symbol"] = sym
    try:
        response = session.get(url,params=base, timeout=10, verify=False)

        if response.status_code == 200:
            api_data = response.json()
            data = api_data["data"]
            # print(len(data))
            for d in data:
                d["symbol"] = sym
            if data:
                report_list.extend(data)
        else:
            print(f"Error Code: {response.status_code}")
        
        time.sleep(.2)
            
    except requests.exceptions.RequestException as error:
        print(f"Network Connection Failed: {error}")

#pd.DataFrame(report_list).to_csv("ANNUAL_REPORT_LINKS.csv")

In [ ]:
pdf_path = r"ANNUAL_REPORT_LINKS.csv"
df1 = pd.read_csv(pdf_path)
df1.head(10)

In [ ]:
df1.info()

In [39]:
cond_2026 = df1.toYr == 2024
df26 = df1[cond_2026]
df26.head(5)

,companyName,fromYr,toYr,submission_type,broadcast_dttm,disseminationDateTime,timeTaken,fileName,attFileSize,symbol
1,360 ONE WAM LIMITED,2023,2024,-,-,-,::,https://nsearchives.nseindia.com/annual_report...,NaN,360ONE
7,3M India Limited,2023,2024,-,-,-,::,https://nsearchives.nseindia.com/annual_report...,NaN,3MINDIA
23,ABB India Limited,2024,2024,New,09-APR-2025 19:57:14,09-APR-2025 19:57:15,00:00:01,https://nsearchives.nseindia.com/annual_report...,NaN,ABB
41,ACC Limited,2023,2024,-,29-MAY-2024 17:45:59,29-MAY-2024 18:05:07,00:19:08,https://nsearchives.nseindia.com/annual_report...,NaN,ACC
59,AIA Engineering Limited,2023,2024,-,14-AUG-2024 19:27:23,14-AUG-2024 19:27:26,00:00:03,https://nsearchives.nseindia.com/annual_report...,NaN,AIAENG


##REPORTS FOR 2025 and 2026

In [ ]:
import os
import requests

DOWNLOAD_FOLDER = r"E:\ANNUAL_REPORTS_2024"
s = requests.Session()
s.headers.update(headers)
s.get("https://nseindia.com")

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

for idx, row in df26.iterrows():
    url = row["fileName"]
    filename = f"{DOWNLOAD_FOLDER}/{row["symbol"]}.pdf"

    print(f"Downloading {filename}...")
    res = s.get(url)
    with open(filename, "wb") as f:
        f.write(res.content)

    if not res.content.startswith(b"%PDF"):
        print("Corrupt or blocked. Deleting...")
        os.remove(filename)
    else:
        print("Saved!")
    time.sleep(.5)
